# Reading data

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

# COPYING BRONZE SCHEMA
df_bronze = spark.read.format("parquet")\
    .option("inferSchema", "true")\
    .option("header", "true")\
    .load("abfss://bronze@adlscarproject.dfs.core.windows.net/rawdata")

# TRANSFORMATION - AUTO INCREMENT, SPLIT, CAST, AND MD5 - HASH FUNCTIONS
df_silver = df_bronze\
        .withColumn('Model_Category', split(col('Model_ID'),'-')[0])\
        .withColumn('End_Date',lit(None).cast(StringType()))\
        .withColumn('isActive',lit(1))\
        .withColumn("Hash_Value", md5(col("Model_Category")))\
        .select('Model_ID','Model_Category','End_Date','isActive','Hash_Value')\
        .dropDuplicates(['Model_ID'])\
        .sort("Model_ID")

# SHOW SCHEMA
df_silver.printSchema()

In [0]:
from pyspark.sql.functions import col
df_bronze.filter(col('model_id').like('Mer-M%')).sort("TimeStamp",ascending=False).display()


In [0]:
from pyspark.sql.functions import col

# df_silver.sort("TimeStamp",ascending=False).display()
df_silver.filter(col('model_id').like('Mer-M%')).sort("TimeStamp",ascending=False).display()

In [0]:
%sql
-- SELECT * 
-- FROM parquet.`abfss://bronze@adlscarproject.dfs.core.windows.net/rawdata`
-- order by branch_id desc

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_date

# Define table name
silver_table_name = "Car_Project.Silver.Models"

# Create the table if it doesn't exist
if not spark.catalog.tableExists(silver_table_name):
    spark.sql(f"""
        CREATE TABLE {silver_table_name} (
            Model_Key BIGINT GENERATED ALWAYS AS IDENTITY,
            Model_ID STRING,
            Model_Category STRING,
            End_Date DATE,
            isActive INT,
            Hash_Value STRING
        )
        USING DELTA
    """)
    print("✅ Table created.")

# Filter to only active records for merge matching
df_active = df_silver.filter("isActive = 1")

# Load existing delta table
delta_silver = DeltaTable.forName(spark, silver_table_name)

# Step 1: Set old records to inactive when a change is detected
delta_silver.alias("target").merge(
    df_active.alias("source"),
    "target.Model_ID = source.Model_ID AND target.isActive = 1 AND target.Hash_Value <> source.Hash_Value"
).whenMatchedUpdate(set={
    "isActive": "0",
    "End_Date": "current_date()"
}).execute()

# Step 2: Insert new versions of changed rows and new rows
delta_silver.alias("target").merge(
    df_silver.alias("source"),
    "target.Model_ID = source.Model_ID AND target.isActive = 1"
).whenNotMatchedInsert(values={
    "Model_ID": "source.Model_ID",
    "Model_Category": "source.Model_Category",
    "End_Date": "NULL",
    "isActive": "1",
    "Hash_Value": "source.Hash_Value"
}).execute()

print("🔁 SCD Type 2 MERGE completed — historical and current records maintained.")


In [0]:
%sql 
select * from car_project.silver.Models 
where model_id like '%Mer-M%'
order by model_key desc

In [0]:
%sql
#drop table car_project.silver.Models 